# 👁️ Pupil-Limbus Segmentation Model Training on Google Colab 🚀

Welcome to the Google Colab training notebook for the Pupil-Limbus eye tracker!  
By using Google Colab high-performance cloud GPUs (T4, L4, or A100), you can train **up to 20x faster** than on a local CPU.

### 📋 Workflow Overview:
1. ✅ Verify GPU is active
2. 📦 Clone the project code from GitHub *(no ZIP uploads needed!)*
3. 📁 Upload `clinical_data.zip` from Google Drive
4. 🛠️ Install dependencies
5. 🏋️ Run training
6. 📤 Export ONNX and download your trained model


In [ ]:
# @title 1. Verify GPU Acceleration 🏎️
# If no GPU found: Runtime -> Change runtime type -> GPU (T4 is free) -> Save
import torch
if torch.cuda.is_available():
    d = torch.cuda.get_device_properties(0)
    print(f'✅ GPU active: {torch.cuda.get_device_name(0)}')
    print(f'   Total GPU memory: {d.total_memory / (1024**3):.2f} GB')
    print(f'   CUDA version:     {torch.version.cuda}')
else:
    print('❌ GPU is NOT active!')
    print('   Go to: Runtime -> Change runtime type -> GPU (T4 GPU is free) -> Save')
    print('   Then restart the runtime and run all cells again.')


## 2. Clone Project Code from GitHub 📦

This pulls the **latest committed code** directly from GitHub.  
No ZIP files needed, no nested folder issues!


In [ ]:
# @title 2. Clone / Update Project from GitHub 🔄
import os, shutil
from pathlib import Path

# CRITICAL: always reset cwd to /content FIRST.
# If a previous cell ran  %cd /content/workspace  and that folder was then deleted,
# git clone fails with: getcwd: cannot access parent directories: No such file or directory
os.chdir('/content')
print(f'Working directory: {os.getcwd()}')

WORKSPACE = Path('/content/workspace')
REPO_URL  = 'https://github.com/Prince649294u83/Centration.git'

if WORKSPACE.exists() and (WORKSPACE / '.git').exists():
    print('🔄 Workspace already cloned - pulling latest changes...')
    !git -C /content/workspace pull
else:
    if WORKSPACE.exists():
        print('🗑️  Removing stale workspace...')
        shutil.rmtree(WORKSPACE)
    print(f'📥 Cloning from {REPO_URL} ...')
    !git clone https://github.com/Prince649294u83/Centration.git /content/workspace

# Verify critical files exist
print('\n📂 Verifying project structure...')
ok = True
for f in ['scripts/train_model.py', 'scripts/export_onnx.py', 'pupil_tracking']:
    path = WORKSPACE / f
    if path.exists():
        print(f'   ✅ {f}')
    else:
        print(f'   ❌ MISSING: {f}')
        ok = False

if ok:
    print('\n✅ Project code is ready!')
else:
    print('\n❌ Some files are missing. Check the repository URL and try again.')


## 3. Connect Your Clinical Data 📁

Your `clinical_data` folder contains annotated training images and masks.  
Choose **one** of the two options below:

- **Option A** *(Recommended)*: Mount Google Drive - upload `clinical_data.zip` once, reuse across sessions.
- **Option B**: Upload `clinical_data.zip` directly to Colab each session.


In [ ]:
# @title Option A: Mount Google Drive & Copy Clinical Data 📁 (Recommended)
# 1. Upload clinical_data.zip to the root of your Google Drive.
# 2. Run this cell.
from google.colab import drive
import shutil
from pathlib import Path

try:
    drive.mount('/content/drive')
    print('✅ Google Drive mounted!')

    drive_zip = Path('/content/drive/MyDrive/clinical_data.zip')
    local_zip = Path('/content/clinical_data.zip')

    if drive_zip.exists():
        print('📋 Copying clinical_data.zip from Drive...')
        shutil.copy2(drive_zip, local_zip)
        mb = local_zip.stat().st_size / 1024 / 1024
        print(f'   ✅ Copied! ({mb:.1f} MB)')
    else:
        print('ℹ️  clinical_data.zip not found in Google Drive root.')
        print('   Please upload it to: My Drive/clinical_data.zip')

except Exception as e:
    print(f'⚠️  Google Drive mount failed: {e}')
    print('   Use Option B (direct upload) instead.')


In [ ]:
# @title Option B: Upload clinical_data.zip Directly 📤 (Alternative)
# Click 'Choose Files', select your local clinical_data.zip, wait for upload.
from google.colab import files
print('📤 Select your clinical_data.zip file to upload...')
uploaded = files.upload()
for name, data in uploaded.items():
    dest = f'/content/{name}'
    with open(dest, 'wb') as f:
        f.write(data)
    mb = len(data) / 1024 / 1024
    print(f'   ✅ Uploaded: {dest} ({mb:.1f} MB)')


In [ ]:
# @title Extract Clinical Data into Workspace 🔓
# Run this AFTER choosing Option A or B above.
import zipfile, shutil
from pathlib import Path

WORKSPACE = Path('/content/workspace')
data_zip  = Path('/content/clinical_data.zip')
data_dest = WORKSPACE / 'clinical_data'

if not data_zip.exists():
    print('❌ clinical_data.zip not found at /content/clinical_data.zip')
    print('   Please run Option A or Option B above first.')
else:
    if data_dest.exists():
        print('🗑️  Removing old clinical_data folder...')
        shutil.rmtree(data_dest)

    mb = data_zip.stat().st_size / 1024 / 1024
    print(f'📦 Extracting clinical_data.zip ({mb:.1f} MB)...')
    with zipfile.ZipFile(data_zip, 'r') as zf:
        # Auto-detect nesting: if zip already has clinical_data/ as root
        members = zf.namelist()
        roots   = set(m.split('/')[0] for m in members if m.split('/')[0])
        if len(roots) == 1 and 'clinical_data' in roots:
            zf.extractall(WORKSPACE)
            print("   ✅ Extracted (detected 'clinical_data/' root in zip)")
        else:
            zf.extractall(data_dest)
            print('   ✅ Extracted into workspace/clinical_data/')

    print('\n📂 Checking data structure...')
    checks = {
        'clinical_data/':                              data_dest,
        'clinical_data/training_data/images/':         data_dest / 'training_data' / 'images',
        'clinical_data/training_data/masks/':          data_dest / 'training_data' / 'masks',
        'clinical_data/annotations/annotations.json':  data_dest / 'annotations' / 'annotations.json',
    }
    all_ok = True
    for label, path in checks.items():
        if path.exists():
            print(f'   ✅ {label}')
        else:
            print(f'   ❌ MISSING: {label}')
            all_ok = False

    if all_ok:
        img_dir  = data_dest / 'training_data' / 'images'
        n_images = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
        print(f'\n✅ Clinical data ready! Found {n_images} training images.')
    else:
        print('\n⚠️  Some expected folders are missing. Check your zip structure.')


## 4. Install Dependencies 🛠️


In [ ]:
# @title Install Core Libraries & Dependencies 🚀
# Takes ~60 seconds on first run.
import os, sys
os.chdir('/content/workspace')
sys.path.insert(0, '/content/workspace')

print('📦 Installing dependencies...')
!pip install -q segmentation-models-pytorch albumentations==1.4.3 onnxruntime-gpu timm

print('\n🔍 Verifying imports...')
import numpy as np
print(f'   ✅ numpy {np.__version__}')
import cv2
print(f'   ✅ opencv {cv2.__version__}')
import albumentations as A
print(f'   ✅ albumentations {A.__version__}')
import segmentation_models_pytorch as smp
print(f'   ✅ smp {smp.__version__}')
import torch
print(f'   ✅ torch {torch.__version__} (CUDA: {torch.cuda.is_available()})')
from pupil_tracking.ml.trainer import Trainer
print('   ✅ pupil_tracking.ml.trainer')
print('\n✅ All packages imported and ready!')


## 5. Run Training Pipeline 🏋️

The production-grade training loop includes:
- ⚡ **Mixed-precision training (AMP)** - doubles GPU throughput
- 📉 **Cosine annealing LR schedule** - finds optimal minima
- 🛑 **Early stopping** - prevents overfitting
- ⚖️ **Inverse-frequency weighted loss** - handles class imbalance

**Expected training time on T4 GPU**: ~15-30 minutes for 300 epochs


In [ ]:
# @title Start GPU-Accelerated Training 🚀
# --num-classes 3  ->  background / pupil / iris
# --num-classes 4  ->  adds suction_ring class
import os
os.chdir('/content/workspace')

!python scripts/train_model.py \\
    --epochs 300 \\
    --batch-size 16 \\
    --lr 0.0005 \\
    --num-classes 3 \\
    --annotation-path clinical_data/annotations/annotations.json \\
    --image-dir clinical_data/training_data/images \\
    --mask-dir clinical_data/training_data/masks \\
    --save-dir models \\
    --device cuda


## 6. Export to ONNX & Download 📤

Convert your trained `.pth` model to optimised ONNX.  
ONNX runs in a **50 MB runtime with no PyTorch dependency** - perfect for the local tracking app.


In [ ]:
# @title Export Checkpoint to ONNX 🌟
import os
from pathlib import Path
os.chdir('/content/workspace')

MODEL_PATH  = '/content/workspace/models/best_model.pth'
ONNX_OUTPUT = '/content/workspace/models/onnx/segmentation.onnx'

if not Path(MODEL_PATH).exists():
    print(f'❌ Model not found: {MODEL_PATH}')
    print('   Make sure training completed successfully in step 5.')
else:
    Path(ONNX_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    print('🔄 Converting to ONNX...')
    !python scripts/export_onnx.py --model "{MODEL_PATH}" --output "{ONNX_OUTPUT}"

    if Path(ONNX_OUTPUT).exists():
        mb = Path(ONNX_OUTPUT).stat().st_size / 1024 / 1024
        print(f'\n✅ ONNX model saved: {ONNX_OUTPUT} ({mb:.1f} MB)')
    else:
        print('❌ ONNX export failed. Check the error above.')


In [ ]:
# @title Save Trained Models to Google Drive 💾 (Recommended)
# Saves .pth and .onnx to Drive so they persist after the session ends.
import shutil
from pathlib import Path

DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/pupil_limbus_models')
MODELS_DIR     = Path('/content/workspace/models')

if not Path('/content/drive').exists():
    print('ℹ️  Google Drive not mounted. Run Option A cell first.')
elif not MODELS_DIR.exists():
    print('❌ Models directory not found. Make sure training completed.')
else:
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f'💾 Copying models to Drive: {DRIVE_SAVE_DIR}')
    copied = 0
    for f in MODELS_DIR.rglob('*'):
        if f.is_file() and f.suffix in ('.pth', '.onnx'):
            dest = DRIVE_SAVE_DIR / f.relative_to(MODELS_DIR)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            mb = f.stat().st_size / 1024 / 1024
            print(f'   ✅ {f.name} ({mb:.1f} MB)')
            copied += 1
    print(f'\n✅ {copied} model file(s) saved to Google Drive.')
    print('   Download from Drive to your local project models/ folder.')


In [ ]:
# @title (Alternative) Direct Download as ZIP 💾
import shutil
from google.colab import files
from pathlib import Path

models_dir   = Path('/content/workspace/models')
archive_path = '/content/trained_models_colab'

if not models_dir.exists():
    print('❌ Models folder not found! Ensure training completed successfully.')
else:
    print('📦 Packaging models into ZIP archive...')
    shutil.make_archive(archive_path, 'zip', str(models_dir))
    mb = Path(f'{archive_path}.zip').stat().st_size / 1024 / 1024
    print(f'✅ Archive ready: {archive_path}.zip ({mb:.1f} MB)')
    try:
        files.download(f'{archive_path}.zip')
        print('⬇️  Download started! Extract into your local project models/ folder.')
    except Exception as e:
        print(f'\n⚠️  Auto-download failed: {e}')
        print(f'   Manually download from the left sidebar: {archive_path}.zip')
